# Filtering and aggregation

Most questions a database answers are of the form "how many / how much, grouped by something,
under some condition". SQL expresses that with `WHERE`, `GROUP BY`, `HAVING`, and aggregate
functions. This notebook runs those queries against the normalised OpenFlights database.

## Learning objectives

By the end of this notebook you will be able to:

- filter rows with `WHERE`, `LIKE`, `IN`, and `BETWEEN`;
- sort and limit results with `ORDER BY` and `LIMIT`;
- summarise with `COUNT`, `COUNT(DISTINCT ...)`, `AVG`, `MIN`, and `MAX`;
- group rows with `GROUP BY` and filter groups with `HAVING`;
- use built-in string and numeric functions to shape a query.

## Concept

SQL evaluates a query in a fixed order that is worth memorising: `FROM` chooses the table, `WHERE`
filters rows, `GROUP BY` collects them into groups, aggregate functions collapse each group, and
`HAVING` filters the groups. `SELECT` then projects columns, `ORDER BY` sorts, and `LIMIT` trims.
A common mistake is putting an aggregate condition in `WHERE` — it belongs in `HAVING`.

Aggregates ignore `NULL` values: `AVG` averages only the non-null rows, and `COUNT(column)` counts
non-null values while `COUNT(*)` counts rows. `COUNT(DISTINCT column)` counts unique non-null
values. These small differences change answers, so state which one you mean.

The OpenFlights database has three big tables — `airports`, `airlines`, and `routes` — plus
normalised `countries` and `cities`. Routes reference airports by id, which is what makes the
joins in the next notebook possible.

## Worked example

### Connect and check the tables

The loader must have built `data/raw/flights.db` first:
`python scripts/download_data.py --module 06 && python 06-databases-and-sql/load.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from load import build_database, DEFAULT_DB
from ds_practice import connect_sqlite, query
from ds_practice.paths import data_path

db_path = data_path(DEFAULT_DB)
if not db_path.exists():
    db_path = build_database()
conn = connect_sqlite(db_path)

print("tables:", query(conn, "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")["name"].tolist())
print("airports:", query(conn, "SELECT COUNT(*) AS n FROM airports")["n"].iloc[0])
print("routes  :", query(conn, "SELECT COUNT(*) AS n FROM routes")["n"].iloc[0])

### Filtering rows

`WHERE` keeps the rows that satisfy a condition. `LIKE` does simple pattern matching with `%`,
`IN` tests membership, and `BETWEEN` is inclusive on both ends.

In [ ]:
busy = query(conn, """
    SELECT name, city_id, iata
    FROM airports
    WHERE type = 'airport' AND iata IS NOT NULL
    ORDER BY name
    LIMIT 5
""")
display(busy)

big = query(conn, """
    SELECT name, altitude
    FROM airports
    WHERE altitude BETWEEN 2000 AND 3000
    ORDER BY altitude DESC
    LIMIT 5
""")
display(big)

### Aggregating

Aggregate functions collapse many rows into one number. Here we count airports, count how many
have an IATA code, and find the highest airport in the table.

In [ ]:
summary = query(conn, """
    SELECT
        COUNT(*)                       AS airports,
        COUNT(a.iata)                  AS with_iata,
        COUNT(DISTINCT c.country_id)   AS countries,
        MAX(a.altitude)                AS highest_ft
    FROM airports a
    JOIN cities c ON c.city_id = a.city_id
""")
display(summary)

### Grouping and having

`GROUP BY` produces one row per group; `HAVING` keeps only groups that pass a condition. This is
how we find the cities that are served by the most airports.

In [ ]:
top_cities = query(conn, """
    SELECT c.name AS city, co.name AS country, COUNT(*) AS airports
    FROM airports a
    JOIN cities c    ON c.city_id = a.city_id
    JOIN countries co ON co.country_id = c.country_id
    GROUP BY c.city_id
    HAVING COUNT(*) >= 3
    ORDER BY airports DESC, city
    LIMIT 10
""")
display(top_cities)

## Exercises

1. **Active European airlines.** Count the airlines per country where `active = 'Y'`, return only
   countries with at least five active airlines, and sort descending.
2. **Long names.** List the ten longest airport names (by character length) that contain the word
   `International`, using the `LENGTH` function and `LIKE`.
3. **Services per airport.** Count how many routes depart from each airport, return the top five,
   and explain why `COUNT(*)` on `routes` differs from counting distinct destination airports.

## Limitations

`LIKE` is case-insensitive for ASCII in SQLite by default, but not for non-ASCII text, and it is
not a full-text search. Aggregates ignore `NULL`, so an average can silently drop rows; always
report how many rows contributed. `GROUP BY` on a wide table can be slow, and SQLite builds a
temporary index unless a suitable index exists. Finally, this dataset is a snapshot of published
schedules, not live traffic, so counts describe the file rather than the world.